# पाठ १३ - एजेन्ट मेमोरी


## सेटअप

यो नोटबुकले कसरी **Microsoft Agent Framework** (MAF) प्रयोग गरेर **दिर्घकालीन स्मृति** भएको यात्रा बुकिङ एजेन्ट बनाउने देखाउँछ।

तपाईंले सिक्नुहुनेछ कि विभिन्न प्रकारका एजेन्ट स्मृति — कार्यशील, छोटो अवधि, र लामो अवधि — कसरी एक एजेन्टले कुराकानीहरूमा जानकारी राख्ने र प्रयोग गर्ने तरिकामा प्रभाव पार्छ।

**पूर्वआवश्यकताहरू:**
- तैनाथ गरिएको च्याट मोडेल भएको Microsoft Foundry परियोजना (जस्तै `gpt-5-mini`)।
- Azure CLI मा लग इन — आफ्नो टर्मिनलमा `az login` चलाउनुहोस्।
- `AZURE_AI_PROJECT_ENDPOINT` — तपाईंको Microsoft Foundry परियोजना अन्तबिन्दु।
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — तपाईंको तैनाथ गरिएको मोडेलको नाम।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## एजेन्ट मेमोरीका प्रकारहरू

AI एजेन्टहरूले विभिन्न प्रकारका मेमोरीहरू उपयोग गर्न सक्छन्, प्रत्येकले फरक उद्देश्य पूरा गर्दछ:

### कार्य स्मृति (Working Memory)
संवाद थ्रेड आफू — एउटै सत्रमा आदान प्रदान गरिएका सन्देशहरू। एजेन्टले सँगैको ठेगानामा पहिलेका सन्देशहरू सन्दर्भ गर्न सक्छ निरन्तरता कायम राख्न। MAF मा यो **`agent.create_session()`** मार्फत सिर्जना गरिन्छ, जसले `AgentSession` फर्काउँछ।

### छोटो अवधि स्मृति (Short-Term Memory)
जानकारी जुन कुनै कार्य वा सत्रको अवधिभर कायम रहन्छ तर स्थायी रूपमा भण्डारण हुँदैन। उदाहरणका लागि, एजेन्टले धेरै चरणको योजना बनाउने संवादमा तथ्यहरू सङ्कलन गर्न सक्छ र अन्तिम यात्रा तालिका तयार पार्न प्रयोग गर्न सक्छ।

### दीर्घकालीन स्मृति (Long-Term Memory)
प्राथमिकताहरू र तथ्यहरू जुन **सत्रहरू बीच** कायम रहन्छन्। फर्कने प्रयोगकर्ताले आफ्नो भोजन प्रतिबन्ध वा यात्रा शैली फेरि दोहोर्याउन नपरोस्। दीर्घकालीन स्मृतिलाई सामान्यतः बाह्य भण्डार — डाटाबेस, फाइल, वा भेक्टर इन्डेक्स — द्वारा समर्थन गरिन्छ र उपकरणमार्फत एजेन्टलाई प्रस्तुत गरिन्छ।


## सेसनहरुसँग कार्यक्षण स्मृति

स्मृतिको सबैभन्दा सरल स्वरूप हो वार्तालाप सेशन। जब तपाईं एउटै सेशन (`agent.create_session()` मार्फत सिर्जना गरिएको) लगातार `agent.run()` कलहरूमा पास गर्नुहुन्छ, एजेन्टले त्यो वार्तालापको सम्पूर्ण इतिहास देख्छ र पहिलेका विवरणहरू सम्झन सक्छ।

आउनुहोस् एउटा यात्रा एजेन्ट बनाउँयौं र कार्यक्षण स्मृतिको प्रदर्शन गरौं।


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

एजेण्टले बजेट ठीकसँग सम्झियो किनभने दुवै सन्देश एउटै सेसन साझा गर्छन्। यसलाई **कार्य स्मृति** भनिन्छ — यो केवल सेसनको जीवनकालका लागि मात्र अवस्थित हुन्छ।

### नयाँ थ्रेडसँग के हुन्छ?

यदि हामीले **नयाँ** सेसन सिर्जना गर्छौं भने, एजेण्टले अघिल्लो कुराकानीको कुनै स्मृति हुँदैन:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## लामो-अवधि मेमोरी नमुना

प्रयोगकर्ता प्राथमिकताहरूलाई **सेसनहरू बीच** सम्झनका लागि, हामीलाई एक दिर्घकालीन भण्डारण चाहिन्छ जुन कुराकानी थ्रेडको बाहिर बस्छ। एजेन्टले यो भण्डारणमा पहुँच **उपकरणहरू** मार्फत गर्छ — ती कार्यहरू जुन यसले जानकारी बचत गर्न र पुन: प्राप्त गर्न बोलाउन सक्छ।

तल हामी एक सरल इन-मेमोरी प्राथमिकता भण्डार कार्यान्वयन गर्दछौं (उत्पादनमा तपाईं यसलाई डाटाबेस वा भेक्टर इन्डेक्ससँग समर्थन गर्नुहुनेछ) र यसलाई एजेन्टले प्रयोग गर्न सक्ने उपकरणहरूका रूपमा खोल्दछौं।

### वास्तुकला
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### परिदृश्य १ — पहिलो पटक प्रयोगकर्ताले वार्षिकोत्सवको यात्रा बुक गर्छ

साराह पहिलो पटक भेटघाट गर्छिन्। एजेन्टले उपकरणहरू मार्फत उनको प्राथमिकताहरू भण्डारण गर्नुपर्छ र होटलहरूको सिफारिस गर्न तिनीहरूलाई प्रयोग गर्नुपर्छ।


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### परिदृश्य २ — सारा हप्ताहरू पछि फर्किएकी छिन्

सारा एक **साम्प्रतिक नयाँ थ्रेड** सुरु गर्छिन् (नयाँ सत्रको नक्कल गर्दै)। काम गर्ने स्मृति खाली छ, तर दीर्घकालीन प्राथमिकता स्टोरमा अझै पनि उनकी जानकारी छ। एजेन्टले यसलाई पुनःप्राप्त गर्नुपर्छ र सिफारिसहरूलाई व्यक्तिगत बनाउन प्रयोग गर्नुपर्छ।


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## सारांश

यस पाठमा तपाइँले एजेन्ट स्मृतिका तीन प्रकारहरू र तिनीहरूलाई Microsoft Agent Framework सँग कसरी कार्यान्वयन गर्ने सिक्नुभयो:

| स्मृति प्रकार | MAF मेकानिज्म | आयु अवधि |
|---|---|---|
| **कार्यरत** | `agent.create_session()` | एकल वार्तालाप |
| **छोटो अवधि** | एउटै थ्रेड भित्र सञ्चित सन्दर्भ | एकल कार्य / सत्र |
| **दीर्घकालीन** | `@tool` कार्यहरू मार्फत पहुँच गरिएको बाह्य संग्रह | सत्रहरू बीच |

### मुख्य बुँदाहरू
1. **`agent.create_session()`** कार्यरत स्मृति प्रदान गर्छ — एजेन्टले सत्र भित्र सम्पूर्ण वार्तालाप इतिहास देख्छ।
2. **नयाँ सत्रहरूले सन्दर्भ गुमाउँछन्** — दीर्घकालीन स्मृति बिना एजेन्टले विगत वार्तालाप सम्झन सक्दैन।
3. **`@tool` कार्यहरू** अन्तर पूर्ति गर्छन् — तिनीहरूले एजेन्टलाई स्थायी स्टोरबाट जानकारी बचत र पुनःप्राप्ति गर्न दिन्छन्।
4. **व्यक्तिकरण समयसँग सुधार हुन्छ** — जे जति बढी रुचिहरू बचत हुन्छन्, एजेन्टका सिफारिसहरू त्यति राम्रो हुन्छन्।

### वास्तविक विश्वका अनुप्रयोगहरू
- **ग्राहक सेवा**: ग्राहक इतिहास र रुचिहरू सम्झनुहोस्
- **व्यक्तिगत सहायकहरू**: दिन वा हप्तामा सन्दर्भ कायम राख्नुहोस्
- **स्वास्थ्य सेवा**: बिरामी जानकारी र रुचिहरू ट्र्याक गर्नुहोस्
- **ई-वाणिज्य**: इतिहासको आधारमा व्यक्तिगत खरिदारी

### अगाडिका कदमहरू
- इन-मेमोरी शब्दकोशलाई डेटाबेस वा भेक्टर स्टोर (जस्तै Azure AI Search) सँग प्रतिस्थापन गर्नुहोस्
- समय-संवेदनशील जानकारीका लागि स्मृति समाप्ति थप्नुहोस्
- साझा स्मृति सहित बहु-एजेन्ट प्रणालीहरू निर्माण गर्नुहोस्
- ज्ञान-ग्राफ-समर्थित स्मृतिका लागि Cognee नोटबुक अन्वेषण गर्नुहोस्


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
